# Image Classification with PyTorch

A progressive exploration of image classification techniques using PyTorch, covering:

1. **Simple CNN** — handwritten digit classification on MNIST
2. **AlexNet & ResNet** — multi-class classification on CIFAR-10
3. **Data Augmentation** — improving generalization with augmentation strategies
4. **Transfer Learning** — off-the-shelf feature extraction and fine-tuning with ResNet-50

---


## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import _LRScheduler
import torch.utils.data as data
from torch.utils.data import DataLoader

import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torchvision.models as models
from torchvision.datasets import MNIST, ImageFolder
from torchvision.datasets import CIFAR10

from sklearn import decomposition, manifold
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import sklearn.metrics as metrics
from sklearn.svm import SVC

from PIL import Image
import copy
import random
import time
import os

## Device & Reproducibility

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## Shared Utilities

Helpers reused across all experiments.

In [ ]:
# ── Checkpoint utilities ──────────────────────────────────────────────────

def save_checkpoint(optimizer, model, epoch, filename):
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    torch.save({
        'optimizer': optimizer.state_dict(),
        'model': model.state_dict(),
        'epoch': epoch
    }, filename)

def load_checkpoint(optimizer, model, filename):
    checkpoint = torch.load(filename)
    model.load_state_dict(checkpoint['model'])
    if optimizer is not None:
        optimizer.load_state_dict(checkpoint['optimizer'])
    return checkpoint['epoch']


# ── Parameter counter ─────────────────────────────────────────────────────

def count_parameters(model):
    total = sum(np.prod(p.shape) for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Total parameters:     {total:,}')
    print(f'Trainable parameters: {trainable:,}')
    return trainable


# ── Image normalization (for plotting) ────────────────────────────────────

def normalize_image(image):
    image_min = image.min()
    image_max = image.max()
    image.clamp_(min=image_min, max=image_max)
    image.add_(-image_min).div_(image_max - image_min + 1e-5)
    return image


# ── t-SNE helpers ─────────────────────────────────────────────────────────

def get_tsne(data, n_components=2, n_images=None):
    if n_images is not None:
        data = data[:n_images]
    tsne = manifold.TSNE(n_components=n_components, random_state=0)
    return tsne.fit_transform(data)

def plot_representations(data, labels, classes=None, n_images=None):
    if n_images is not None:
        data = data[:n_images]
        labels = labels[:n_images]
    fig, ax = plt.subplots(figsize=(10, 10))
    scatter = ax.scatter(data[:, 0], data[:, 1], c=labels, cmap='tab10')
    handles, leg_labels = scatter.legend_elements()
    legend_labels = classes if classes else leg_labels
    ax.legend(handles=handles, labels=legend_labels)
    plt.tight_layout()
    plt.show()


# ── Confusion matrix ──────────────────────────────────────────────────────

def plot_confusion_matrix(labels, pred_labels, classes=None):
    fig, ax = plt.subplots(figsize=(10, 10))
    cm = ConfusionMatrixDisplay(
        confusion_matrix(labels, pred_labels),
        display_labels=classes
    )
    cm.plot(values_format='d', cmap='Blues', ax=ax)
    if classes:
        plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()


# ── Most-incorrect samples ────────────────────────────────────────────────

def get_predictions(model, loader, device, color=False):
    model.eval()
    images, labels, preds = [], [], []
    with torch.no_grad():
        for batch, targets in loader:
            batch = batch.to(device)
            out, _ = model(batch)
            probs = F.softmax(out, dim=1)
            images.append(batch.cpu())
            labels.append(targets.cpu())
            preds.append(probs.cpu())
    return torch.cat(images), torch.cat(labels), torch.cat(preds)

def plot_most_incorrect(incorrect, n_images, classes=None, color=False):
    rows = cols = int(np.sqrt(n_images))
    fig = plt.figure(figsize=(20, 10))
    for i in range(rows * cols):
        ax = fig.add_subplot(rows, cols, i + 1)
        image, true_label, probs = incorrect[i]
        true_prob = probs[true_label]
        incorrect_prob, incorrect_label = torch.max(probs, dim=0)
        if color:
            img = normalize_image(image.permute(1, 2, 0))
            ax.imshow(img.numpy())
        else:
            ax.imshow(image.view(image.shape[-2], image.shape[-1]).numpy(), cmap='bone')
        tl = classes[true_label] if classes else str(true_label.item())
        il = classes[incorrect_label] if classes else str(incorrect_label.item())
        ax.set_title(f'true: {tl} ({true_prob:.2f})\npred: {il} ({incorrect_prob:.2f})')
        ax.axis('off')
    fig.subplots_adjust(hspace=0.5)
    plt.show()


# ── Generic training loop ─────────────────────────────────────────────────

def run_training(model, optimizer, criterion, train_loader, valid_loader,
                 valid_set, first_epoch, num_epochs, checkpoint_prefix='ckpt'):
    train_losses, valid_losses = [], []
    for epoch in range(first_epoch, first_epoch + num_epochs):
        model.train()
        batch_losses = []
        for batch, targets in train_loader:
            batch, targets = batch.to(device), targets.to(device)
            optimizer.zero_grad()
            predictions, _ = model(batch)
            loss = criterion(predictions, targets)
            loss.backward()
            optimizer.step()
            batch_losses.append(loss.item())
        train_loss = np.mean(batch_losses)

        model.eval()
        val_losses, y_pred = [], []
        with torch.no_grad():
            for batch, targets in valid_loader:
                batch, targets = batch.to(device), targets.to(device)
                predictions, _ = model(batch)
                val_losses.append(criterion(predictions, targets).item())
                y_pred.extend(predictions.argmax(dim=1).cpu().numpy())
        valid_loss = np.mean(val_losses)
        accuracy = (np.array(y_pred) == np.array(valid_set.targets)).mean()

        print(f'[{epoch:03d}] train loss: {train_loss:.4f}  '
              f'val loss: {valid_loss:.4f}  val acc: {accuracy*100:.2f}%')
        train_losses.append(train_loss)
        valid_losses.append(valid_loss)
        save_checkpoint(optimizer, model, epoch,
                        f'checkpoints/{checkpoint_prefix}-{epoch:03d}.pkl')
    return train_losses, valid_losses


def plot_learning_curves(train_losses, valid_losses):
    epochs = range(1, len(train_losses) + 1)
    plt.figure(figsize=(10, 6))
    plt.plot(epochs, train_losses, label='Training loss')
    plt.plot(epochs, valid_losses, label='Validation loss')
    plt.legend()
    plt.title('Learning curves')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.xticks(epochs)
    plt.tight_layout()
    plt.show()

---
## Part 1 — Simple CNN on MNIST

A two-layer CNN trained to classify handwritten digits (10 classes).
Architecture: Conv → Conv → MaxPool → Dropout → FC → Dropout → FC.


### Model

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_channels=1, num_classes=10):
        super().__init__()
        self.conv1  = nn.Conv2d(num_channels, 32, 3, stride=1, padding=1)
        self.conv2  = nn.Conv2d(32, 32, 3, stride=1, padding='same')
        self.pool1  = nn.MaxPool2d(2, stride=2)
        self.drop1  = nn.Dropout(p=0.25)
        self.fc1    = nn.Linear(14 * 14 * 32, 128)
        self.drop2  = nn.Dropout(p=0.5)
        self.fc2    = nn.Linear(128, num_classes)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.drop1(self.pool1(x))
        x = x.reshape(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.drop2(x)
        feats = x
        x = self.fc2(x)
        return x, feats

# Quick sanity check
torch.manual_seed(SEED)
_out, _ = SimpleCNN()(torch.rand(1, 1, 28, 28))
print('Output shape:', _out.shape)  # [1, 10]

### Dataset — MNIST

In [ ]:
mnist_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.1307], [0.3081])
])

mnist_train = MNIST('./data/mnist', train=True,  download=True, transform=mnist_transform)
mnist_valid = MNIST('./data/mnist', train=False, download=True, transform=mnist_transform)

mnist_train_loader = DataLoader(mnist_train, batch_size=256, shuffle=True,  num_workers=0)
mnist_valid_loader = DataLoader(mnist_valid, batch_size=512, shuffle=False, num_workers=0)

print(f'Train: {mnist_train.data.shape}  |  Valid: {mnist_valid.data.shape}')

# Preview first 64 digits
sample = mnist_train.data[:64].reshape(8, 8, 28, 28).permute(0, 2, 1, 3).reshape(8*28, 8*28)
plt.figure(figsize=(8, 8))
plt.imshow(sample, cmap='gray')
plt.axis('off')
plt.title('First 64 MNIST digits')
plt.show()

### Train SimpleCNN

In [ ]:
model_cnn = SimpleCNN().to(device)
count_parameters(model_cnn)

criterion_cnn = nn.CrossEntropyLoss()
optimizer_cnn = optim.SGD(model_cnn.parameters(), lr=0.01, momentum=0.9, nesterov=True)

train_losses_cnn, valid_losses_cnn = run_training(
    model_cnn, optimizer_cnn, criterion_cnn,
    mnist_train_loader, mnist_valid_loader,
    mnist_valid,
    first_epoch=1, num_epochs=10,
    checkpoint_prefix='mnist'
)

plot_learning_curves(train_losses_cnn, valid_losses_cnn)

### Error Analysis — MNIST

In [ ]:
images_m, labels_m, probs_m = get_predictions(model_cnn, mnist_valid_loader, device)
pred_labels_m = torch.argmax(probs_m, dim=1)

print(f'Validation errors: {(pred_labels_m != labels_m).sum().item()} / {len(mnist_valid)}')

# Confusion matrix
plot_confusion_matrix(labels_m.numpy(), pred_labels_m.numpy())

# Most-confident wrong predictions
corrects_m = torch.eq(labels_m, pred_labels_m)
incorrect_m = [(img, lbl, prob)
               for img, lbl, prob, ok in zip(images_m, labels_m, probs_m, corrects_m)
               if not ok]
incorrect_m.sort(key=lambda x: torch.max(x[2]).item(), reverse=True)
plot_most_incorrect(incorrect_m, n_images=25)

### t-SNE of Learned Representations

In [ ]:
def get_representations_cnn(model, loader, device):
    model.eval()
    outputs, intermediates, labels = [], [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            out, feats = model(x)
            outputs.append(out.cpu())
            intermediates.append(feats.cpu())
            labels.append(y)
    return torch.cat(outputs), torch.cat(intermediates), torch.cat(labels)

outputs_m, intermediates_m, labels_m_tsne = get_representations_cnn(
    model_cnn, mnist_valid_loader, device)

N = 5000
print('t-SNE of output logits:')
plot_representations(get_tsne(outputs_m, n_images=N), labels_m_tsne, n_images=N)

print('t-SNE of penultimate layer features:')
plot_representations(get_tsne(intermediates_m, n_images=N), labels_m_tsne, n_images=N)

---
## Part 2 — AlexNet & ResNet on CIFAR-10

CIFAR-10 contains 60 000 color images (32×32) across 10 classes.
We implement two architectures:
- **AlexNet** (adapted for 32×32 inputs)
- **BasicResNet** (simplified ResNet with skip connections)


### Dataset — CIFAR-10

In [ ]:
ROOT_CIFAR = '.data/CIFAR10'

_raw = datasets.CIFAR10(root=ROOT_CIFAR, train=True, download=True)
cifar_means = _raw.data.mean(axis=(0, 1, 2)) / 255
cifar_stds  = _raw.data.std(axis=(0, 1, 2))  / 255
print(f'CIFAR-10 means: {cifar_means}')
print(f'CIFAR-10 stds:  {cifar_stds}')

cifar_train_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=cifar_means, std=cifar_stds)
])
cifar_valid_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=cifar_means, std=cifar_stds)
])

cifar_train = datasets.CIFAR10(root=ROOT_CIFAR, train=True,  download=True, transform=cifar_train_tf)
cifar_valid = datasets.CIFAR10(root=ROOT_CIFAR, train=False, download=True, transform=cifar_valid_tf)

BATCH = 256
cifar_train_loader = DataLoader(cifar_train, batch_size=BATCH, shuffle=True)
cifar_valid_loader = DataLoader(cifar_valid, batch_size=BATCH, shuffle=False)

print(f'Train: {len(cifar_train)}  |  Valid: {len(cifar_valid)}')
print('Classes:', cifar_valid.classes)

# Preview 25 samples
N_PREVIEW = 25
imgs_p, lbls_p = zip(*[cifar_train[i] for i in range(N_PREVIEW)])
rows = cols = int(np.sqrt(N_PREVIEW))
fig = plt.figure(figsize=(10, 10))
for i, (img, lbl) in enumerate(zip(imgs_p, lbls_p)):
    ax = fig.add_subplot(rows, cols, i + 1)
    ax.imshow(normalize_image(img.clone()).permute(1, 2, 0).numpy())
    ax.set_title(cifar_valid.classes[lbl])
    ax.axis('off')
plt.tight_layout()
plt.show()

### AlexNet (adapted for 32×32 inputs)

In [ ]:
class AlexNet(nn.Module):
    def __init__(self, output_dim):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 48, 5, 2, 2),   # 32 → 15
            nn.MaxPool2d(2),              # 15 → 7
            nn.ReLU(inplace=True),
            nn.Conv2d(48, 128, 5, 1, 2),
            nn.MaxPool2d(2),              # 7 → 3
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 192, 3, 1, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(192, 192, 3, 1, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(192, 128, 3, 1, 1),
            nn.MaxPool2d(2),              # 3 → 1  (but padding keeps it 2)
            nn.ReLU(inplace=True),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(128 * 2 * 2, 2048),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(2048, 2048),
            nn.ReLU(inplace=True),
            nn.Linear(2048, output_dim),
        )

    def forward(self, x):
        x = self.features(x)
        feats = x.view(x.shape[0], -1)
        return self.classifier(feats), feats

# Sanity check
torch.manual_seed(SEED)
_out, _ = AlexNet(5)(torch.rand(1, 3, 32, 32))
print('Output shape:', _out.shape)  # [1, 5]

### Train AlexNet

In [ ]:
model_alex = AlexNet(output_dim=10).to(device)
count_parameters(model_alex)

criterion_alex = nn.CrossEntropyLoss()
optimizer_alex = optim.SGD(model_alex.parameters(), lr=0.01, momentum=0.9, nesterov=True)

train_losses_alex, valid_losses_alex = run_training(
    model_alex, optimizer_alex, criterion_alex,
    cifar_train_loader, cifar_valid_loader,
    cifar_valid,
    first_epoch=1, num_epochs=15,
    checkpoint_prefix='alexnet'
)

plot_learning_curves(train_losses_alex, valid_losses_alex)

### Error Analysis — AlexNet

In [ ]:
images_a, labels_a, probs_a = get_predictions(model_alex, cifar_valid_loader, device)
pred_labels_a = torch.argmax(probs_a, dim=1)

print(f'Validation errors: {(pred_labels_a != labels_a).sum().item()} / {len(cifar_valid)}')

plot_confusion_matrix(labels_a.numpy(), pred_labels_a.numpy(), classes=cifar_valid.classes)

corrects_a = torch.eq(labels_a, pred_labels_a)
incorrect_a = [(img, lbl, prob)
               for img, lbl, prob, ok in zip(images_a, labels_a, probs_a, corrects_a)
               if not ok]
incorrect_a.sort(key=lambda x: torch.max(x[2]).item(), reverse=True)
plot_most_incorrect(incorrect_a, n_images=36, classes=cifar_valid.classes, color=True)

### Filter Visualization — AlexNet

In [ ]:
def plot_filters(filters, normalize=True):
    filters = filters.cpu()
    n = filters.shape[0]
    rows = cols = int(np.sqrt(n))
    fig = plt.figure(figsize=(20, 10))
    for i in range(rows * cols):
        img = filters[i]
        if normalize:
            img = normalize_image(img.clone())
        ax = fig.add_subplot(rows, cols, i + 1)
        ax.imshow(img.permute(1, 2, 0).numpy())
        ax.axis('off')
    fig.subplots_adjust(wspace=-0.9)
    plt.show()

# Filters from our trained AlexNet
plot_filters(model_alex.features[0].weight.data)

# Filters from ImageNet pretrained AlexNet (for comparison)
pretrained_alex = models.alexnet(pretrained=True)
plot_filters(pretrained_alex.features[0].weight.data)

### BasicResNet with Skip Connections

In [ ]:
class BasicResBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size,
                 stride=1, padding=1, downsample=False):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size,
                               stride, padding, bias=False)
        self.bn1   = nn.BatchNorm2d(out_channels)
        self.relu  = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_channels)

        if downsample:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.downsample = None

    def forward(self, x):
        identity = x
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        if self.downsample is not None:
            identity = self.downsample(identity)
        x += identity
        return self.relu(x)


class BasicResNet(nn.Module):
    def __init__(self, output_dim):
        super().__init__()
        self.layer1  = BasicResBlock(3,  32, 3, stride=2, downsample=True)
        self.layer2  = BasicResBlock(32, 32, 3)
        self.layer3  = BasicResBlock(32, 64, 3, stride=2, downsample=True)
        self.layer4  = BasicResBlock(64, 64, 3)
        self.avgpool = nn.AdaptiveAvgPool2d(2)
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(64 * 2 * 2, output_dim)
        )

    def forward(self, x):
        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))
        x = self.avgpool(x)
        feats = x.view(x.shape[0], -1)
        return self.classifier(feats), feats

# Sanity check
torch.manual_seed(SEED)
_out, _ = BasicResNet(5)(torch.rand(1, 3, 32, 32))
print('Output shape:', _out.shape)  # [1, 5]

### Train BasicResNet

In [ ]:
model_resnet = BasicResNet(output_dim=10).to(device)
count_parameters(model_resnet)

criterion_res = nn.CrossEntropyLoss()
optimizer_res = optim.SGD(model_resnet.parameters(), lr=0.005, momentum=0.9, nesterov=True)

train_losses_res, valid_losses_res = run_training(
    model_resnet, optimizer_res, criterion_res,
    cifar_train_loader, cifar_valid_loader,
    cifar_valid,
    first_epoch=1, num_epochs=15,
    checkpoint_prefix='resnet'
)

plot_learning_curves(train_losses_res, valid_losses_res)

---
## Part 3 — Data Augmentation on CIFAR-10

We compare several augmentation strategies and their effect on training.
Common techniques: `RandomHorizontalFlip`, `RandomCrop`, `ColorJitter`, `RandomRotation`.


### Visualizing Augmentation Effects

In [ ]:
# Load a raw (un-normalized) CIFAR-10 sample for visualization
_raw_cifar = datasets.CIFAR10(root=ROOT_CIFAR, train=True, download=False)
_sample_img = Image.fromarray(_raw_cifar.data[0])

augmentations = {
    'Original': transforms.Compose([transforms.ToTensor()]),
    'HFlip': transforms.Compose([transforms.RandomHorizontalFlip(p=1.0), transforms.ToTensor()]),
    'RandomCrop': transforms.Compose([transforms.Pad(4), transforms.RandomCrop(32), transforms.ToTensor()]),
    'ColorJitter': transforms.Compose([
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
        transforms.ToTensor()
    ]),
    'Rotation': transforms.Compose([transforms.RandomRotation(15), transforms.ToTensor()]),
    'Combined': transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.Pad(4),
        transforms.RandomCrop(32),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor()
    ]),
}

fig, axes = plt.subplots(1, len(augmentations), figsize=(18, 3))
for ax, (name, tf) in zip(axes, augmentations.items()):
    img_t = tf(_sample_img)
    ax.imshow(img_t.permute(1, 2, 0).clamp(0, 1).numpy())
    ax.set_title(name)
    ax.axis('off')
plt.suptitle('Augmentation Examples', fontsize=14)
plt.tight_layout()
plt.show()

### Train ResNet with Data Augmentation

In [ ]:
# Training transform: adds random flip + crop
augmented_train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.Pad(4),
    transforms.RandomCrop(32),
    transforms.ToTensor(),
    transforms.Normalize(mean=cifar_means, std=cifar_stds)
])

cifar_train_aug = datasets.CIFAR10(root=ROOT_CIFAR, train=True,
                                   download=False, transform=augmented_train_tf)
cifar_train_aug_loader = DataLoader(cifar_train_aug, batch_size=BATCH, shuffle=True)

# Fresh ResNet instance
model_aug = BasicResNet(output_dim=10).to(device)
optimizer_aug = optim.SGD(model_aug.parameters(), lr=0.005, momentum=0.9, nesterov=True)

train_losses_aug, valid_losses_aug = run_training(
    model_aug, optimizer_aug, nn.CrossEntropyLoss(),
    cifar_train_aug_loader, cifar_valid_loader,
    cifar_valid,
    first_epoch=1, num_epochs=15,
    checkpoint_prefix='resnet_aug'
)

plot_learning_curves(train_losses_aug, valid_losses_aug)

### Compare: Baseline vs Augmented

In [ ]:
epochs = range(1, len(train_losses_res) + 1)
plt.figure(figsize=(10, 5))
plt.plot(epochs, valid_losses_res, label='ResNet (no augmentation)')
plt.plot(epochs, valid_losses_aug, label='ResNet (with augmentation)')
plt.legend()
plt.title('Validation Loss: Baseline vs Augmented')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.tight_layout()
plt.show()

---
## Part 4 — Transfer Learning with ResNet-50

We explore two transfer learning approaches on a small binary classification dataset:

1. **Off-the-shelf**: use frozen pretrained features + linear SVM classifier
2. **Fine-tuning**: unfreeze and retrain the whole network

> **Dataset**: Download the Alien vs Predator
> dataset and extract it to `data/avp/`.


### Dataset — Alien vs Predator

In [ ]:
# ImageNet normalization statistics (standard for pretrained torchvision models)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

avp_train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomCrop(192),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

avp_valid_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.CenterCrop(192),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

avp_train = ImageFolder(root='data/avp/train', transform=avp_train_tf)
avp_valid = ImageFolder(root='data/avp/valid', transform=avp_valid_tf)

avp_train_loader = DataLoader(avp_train, batch_size=64, shuffle=True)
avp_valid_loader = DataLoader(avp_valid, batch_size=64, shuffle=False)

print(f'Train: {len(avp_train)}  |  Valid: {len(avp_valid)}')
print('Classes:', avp_train.classes)

### Train from Scratch (Baseline)

In [ ]:
def avp_run_training(model, optimizer, train_loader, valid_loader,
                     first_epoch, num_epochs, checkpoint_prefix):
    criterion = nn.CrossEntropyLoss()
    train_losses, valid_losses = [], []
    for epoch in range(first_epoch, first_epoch + num_epochs):
        model.train()
        batch_losses = []
        for batch, targets in train_loader:
            batch, targets = batch.to(device), targets.to(device)
            optimizer.zero_grad()
            out = model(batch)
            # handle models that return a tuple
            if isinstance(out, tuple):
                out = out[0]
            loss = criterion(out, targets)
            loss.backward()
            optimizer.step()
            batch_losses.append(float(loss.item()))
        train_loss = np.mean(batch_losses)

        model.eval()
        val_losses, y_true, y_pred = [], [], []
        with torch.no_grad():
            for batch, targets in valid_loader:
                batch, targets = batch.to(device), targets.to(device)
                out = model(batch)
                if isinstance(out, tuple):
                    out = out[0]
                val_losses.append(float(criterion(out, targets).item()))
                y_true.extend(targets.cpu().numpy())
                y_pred.extend(out.argmax(dim=1).cpu().numpy())
        valid_loss = np.mean(val_losses)
        acc = metrics.accuracy_score(y_true, y_pred)

        print(f'[{epoch:02d}] train: {train_loss:.4f}  val: {valid_loss:.4f}  acc: {acc:.4f}')
        train_losses.append(train_loss)
        valid_losses.append(valid_loss)
        save_checkpoint(optimizer, model, epoch,
                        f'checkpoints/{checkpoint_prefix}_{epoch:03d}.pkl')
    return train_losses, valid_losses

# Small ResNet trained from scratch
model_scratch = BasicResNet(output_dim=2).to(device)
opt_scratch = optim.SGD(model_scratch.parameters(), lr=0.01, momentum=0.9)
tl_s, vl_s = avp_run_training(model_scratch, opt_scratch,
                               avp_train_loader, avp_valid_loader,
                               1, 15, 'avp_scratch')
plot_learning_curves(tl_s, vl_s)

### Off-the-Shelf Transfer Learning

In [ ]:
# Load pretrained ResNet-50 and extract features
resnet50 = models.resnet50(pretrained=True).to(device)
resnet50.eval()

def extract_features(model, loader, device):
    """Remove FC, extract penultimate features."""
    model.fc = nn.Sequential()
    feats, labels = [], []
    with torch.no_grad():
        for batch, targets in loader:
            batch = batch.to(device)
            feats.append(model(batch).cpu())
            labels.append(targets)
    return torch.cat(feats), torch.cat(labels)

train_feats, train_lbls = extract_features(resnet50, avp_train_loader, device)
valid_feats, valid_lbls = extract_features(resnet50, avp_valid_loader, device)
print(f'Feature shapes: train {train_feats.shape}, valid {valid_feats.shape}')

# t-SNE of pretrained features
N = 200
tsne_pre = get_tsne(valid_feats.numpy(), n_images=N)
plot_representations(tsne_pre, valid_lbls.numpy(), avp_valid.classes, n_images=N)

# Train a linear SVM on top
svm = SVC(kernel='linear')
svm.fit(train_feats.numpy(), train_lbls.numpy())
y_hat = svm.predict(valid_feats.numpy())
print(f'Off-the-shelf SVM accuracy: {metrics.accuracy_score(valid_lbls, y_hat):.3f}')
print(metrics.classification_report(valid_lbls, y_hat, target_names=avp_valid.classes))

### Fine-Tuning

In [ ]:
# Phase 1: train only the final FC layer
model_ft = models.resnet50(pretrained=True)
model_ft.fc = nn.Linear(model_ft.fc.in_features, 2)
model_ft = model_ft.to(device)

opt_fc = optim.SGD(model_ft.fc.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)
tl_ft, vl_ft = avp_run_training(model_ft, opt_fc,
                                avp_train_loader, avp_valid_loader,
                                first_epoch=1, num_epochs=2, checkpoint_prefix='avp_ft')

# Phase 2: fine-tune the whole network with a smaller lr
opt_full = optim.SGD(model_ft.parameters(), lr=0.001, momentum=0.9)
tl_full, vl_full = avp_run_training(model_ft, opt_full,
                                    avp_train_loader, avp_valid_loader,
                                    first_epoch=3, num_epochs=10, checkpoint_prefix='avp_ft')

tl_all = tl_ft + tl_full
vl_all = vl_ft + vl_full
plot_learning_curves(tl_all, vl_all)

# Load best checkpoint and inspect features
best_epoch = int(np.argmin(vl_all)) + 1
load_checkpoint(opt_full, model_ft, f'checkpoints/avp_ft_{best_epoch:03d}.pkl')
print(f'Best validation loss at epoch {best_epoch}')

# t-SNE of fine-tuned features
model_ft.fc = nn.Sequential()
ft_feats, ft_lbls = extract_features(model_ft, avp_valid_loader, device)
tsne_ft = get_tsne(ft_feats.numpy(), n_images=N)
plot_representations(tsne_ft, ft_lbls.numpy(), avp_valid.classes, n_images=N)

---
## Summary

| Part | Dataset | Model | Technique |
|------|---------|-------|-----------|
| 1 | MNIST | SimpleCNN | Training from scratch |
| 2 | CIFAR-10 | AlexNet, BasicResNet | Architecture comparison |
| 3 | CIFAR-10 | BasicResNet | Data augmentation |
| 4 | Alien vs Predator | ResNet-50 | Transfer learning (off-the-shelf + fine-tuning) |

**Key takeaways:**
- Residual connections allow deeper networks to train more effectively.
- Data augmentation reduces overfitting on small/medium datasets.
- Pretrained features are surprisingly powerful even without fine-tuning.
- Fine-tuning a pretrained network outperforms training from scratch on small datasets.
